In [1]:
import SimpleITK as sitk

# Adjust paths for your patient
patient_id = "Amifer"  # or your test patient
base = f"/home/ayeluru/vascular-superenhancement-4d-flow/working_dir/all_patients/patient_data/{patient_id}/nifti"

cine_orig = sitk.ReadImage(f"{base}/3d_cine_{patient_id}_per_timepoint/3d_cine_{patient_id}_frame_00.nii.gz")
cine_full_fov = sitk.ReadImage(f"{base}/3d_cine_{patient_id}_per_timepoint_full_fov/3d_cine_{patient_id}_frame_00.nii.gz")
flow_ref = sitk.ReadImage(f"{base}/4d_flow_mag_{patient_id}_per_timepoint_full_fov/4d_flow_mag_{patient_id}_frame_00.nii.gz")

print("=== Cine Original (cine space) ===")
print(f"Size: {cine_orig.GetSize()}")
print(f"Spacing: {cine_orig.GetSpacing()}")
print(f"Origin: {cine_orig.GetOrigin()}")

print("\n=== Cine Full FOV (resampled to flow space) ===")
print(f"Size: {cine_full_fov.GetSize()}")
print(f"Spacing: {cine_full_fov.GetSpacing()}")
print(f"Origin: {cine_full_fov.GetOrigin()}")

print("\n=== Flow Reference ===")
print(f"Size: {flow_ref.GetSize()}")
print(f"Spacing: {flow_ref.GetSpacing()}")
print(f"Origin: {flow_ref.GetOrigin()}")

# Check if cine_full_fov matches flow_ref geometry
print("\n=== Geometry Match Check ===")
print(f"Origins match: {cine_full_fov.GetOrigin() == flow_ref.GetOrigin()}")
print(f"Spacings match: {cine_full_fov.GetSpacing() == flow_ref.GetSpacing()}")
print(f"Sizes match: {cine_full_fov.GetSize() == flow_ref.GetSize()}")

=== Cine Original (cine space) ===
Size: (512, 512, 129)
Spacing: (0.7031000256538391, 0.7031000256538391, 1.4000017642974854)
Origin: (-128.03684997558594, -174.96095275878906, 38.275142669677734)

=== Cine Full FOV (resampled to flow space) ===
Size: (256, 256, 152)
Spacing: (1.4062508344650269, 1.4062508344650269, 1.3999019861221313)
Origin: (-143.8280029296875, -172.73399353027344, 22.894100189208984)

=== Flow Reference ===
Size: (256, 256, 152)
Spacing: (1.4062508344650269, 1.4062508344650269, 1.3999019861221313)
Origin: (-143.8280029296875, -172.73399353027344, 22.894100189208984)

=== Geometry Match Check ===
Origins match: True
Spacings match: True
Sizes match: True


In [2]:
import SimpleITK as sitk
import numpy as np

patient_id = "Amifer"
base = f"/home/ayeluru/vascular-superenhancement-4d-flow/working_dir/all_patients/patient_data/{patient_id}/nifti"

# Load the original 4D cine (before splitting)
cine_4d = sitk.ReadImage(f"{base}/3d_cine_{patient_id}.nii.gz")

# Load frame 00 from per_timepoint (original FOV, just split)
cine_frame0 = sitk.ReadImage(f"{base}/3d_cine_{patient_id}_per_timepoint/3d_cine_{patient_id}_frame_00.nii.gz")

# Load frame 00 from per_timepoint_full_fov (resampled to flow space)
cine_full_fov_frame0 = sitk.ReadImage(f"{base}/3d_cine_{patient_id}_per_timepoint_full_fov/3d_cine_{patient_id}_frame_00.nii.gz")

# Load original 4D flow mag
flow_4d = sitk.ReadImage(f"{base}/4d_flow_mag_{patient_id}.nii.gz")

print("=== 4D Cine (original) ===")
print(f"Size: {cine_4d.GetSize()}")
print(f"Origin: {cine_4d.GetOrigin()}")
print(f"Direction: {cine_4d.GetDirection()}")

print("\n=== Cine Frame 0 (split from 4D, no resampling) ===")
print(f"Size: {cine_frame0.GetSize()}")
print(f"Origin: {cine_frame0.GetOrigin()}")
print(f"Direction: {cine_frame0.GetDirection()}")

print("\n=== Cine Full FOV Frame 0 (resampled to flow grid) ===")
print(f"Size: {cine_full_fov_frame0.GetSize()}")
print(f"Origin: {cine_full_fov_frame0.GetOrigin()}")
print(f"Direction: {cine_full_fov_frame0.GetDirection()}")

print("\n=== 4D Flow Mag (original) ===")
print(f"Size: {flow_4d.GetSize()}")
print(f"Origin: {flow_4d.GetOrigin()}")
print(f"Direction: {flow_4d.GetDirection()}")

# Test: Does 4D->3D slicing preserve origin correctly?
print("\n=== Verification: 4D to 3D extraction ===")
slice_test = cine_4d[:,:,:,0]
print(f"4D origin[:3]: {cine_4d.GetOrigin()[:3]}")
print(f"Extracted 3D origin: {slice_test.GetOrigin()}")
print(f"Match: {cine_4d.GetOrigin()[:3] == slice_test.GetOrigin()}")

# Physical location test: Pick a point in cine original space
# and check if it maps to the same location in the resampled version
print("\n=== Physical Point Alignment Test ===")
# Pick center voxel of cine_frame0
center_idx = tuple(s // 2 for s in cine_frame0.GetSize())
world_pt = cine_frame0.TransformIndexToPhysicalPoint(center_idx)
print(f"Center voxel index in cine_frame0: {center_idx}")
print(f"World coordinate: {world_pt}")

# Is this point inside cine_full_fov_frame0?
try:
    idx_in_full_fov = cine_full_fov_frame0.TransformPhysicalPointToIndex(world_pt)
    print(f"Corresponding index in cine_full_fov_frame0: {idx_in_full_fov}")
    
    # Compare intensities
    val_orig = cine_frame0.GetPixel(center_idx)
    val_resampled = cine_full_fov_frame0.GetPixel(idx_in_full_fov)
    print(f"Intensity at original: {val_orig}")
    print(f"Intensity at resampled (nearest index): {val_resampled}")
except Exception as e:
    print(f"Point outside resampled volume: {e}")

=== 4D Cine (original) ===
Size: (512, 512, 129, 20)
Origin: (-128.03684997558594, -174.96095275878906, 38.275142669677734, 0.0)
Direction: (0.9999834856728781, 0.0, -0.005747032054463232, 0.0, 0.0, 1.0, 0.0, 0.0, 0.005747032409918776, 0.0, 0.999983485674921, 0.0, 0.0, 0.0, 0.0, 1.0)

=== Cine Frame 0 (split from 4D, no resampling) ===
Size: (512, 512, 129)
Origin: (-128.03684997558594, -174.96095275878906, 38.275142669677734)
Direction: (0.9999834856728781, 0.0, -0.005747032054463232, 0.0, 1.0, 0.0, 0.005747032409918776, 0.0, 0.999983485674921)

=== Cine Full FOV Frame 0 (resampled to flow grid) ===
Size: (256, 256, 152)
Origin: (-143.8280029296875, -172.73399353027344, 22.894100189208984)
Direction: (1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)

=== 4D Flow Mag (original) ===
Size: (256, 256, 152, 20)
Origin: (-143.8280029296875, -172.73399353027344, 22.894100189208984, 0.0)
Direction: (1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0)

=== Verification:

In [3]:
import SimpleITK as sitk
import numpy as np

patient_id = "Amifer"
base = f"/home/ayeluru/vascular-superenhancement-4d-flow/working_dir/all_patients/patient_data/{patient_id}/nifti"

cine_frame0 = sitk.ReadImage(f"{base}/3d_cine_{patient_id}_per_timepoint/3d_cine_{patient_id}_frame_00.nii.gz")
cine_full_fov = sitk.ReadImage(f"{base}/3d_cine_{patient_id}_per_timepoint_full_fov/3d_cine_{patient_id}_frame_00.nii.gz")

# Sample multiple points and check intensity correlation
np.random.seed(42)
n_samples = 100
correlations = []

cine_arr = sitk.GetArrayFromImage(cine_frame0)
size = cine_frame0.GetSize()

orig_vals = []
resampled_vals = []

for _ in range(n_samples):
    # Random voxel in cine_frame0 (away from edges)
    idx = tuple(np.random.randint(size[i]//4, 3*size[i]//4) for i in range(3))
    world_pt = cine_frame0.TransformIndexToPhysicalPoint(idx)
    
    try:
        # Get corresponding point in resampled image
        idx_resamp = cine_full_fov.TransformPhysicalPointToContinuousIndex(world_pt)
        # Check if inside bounds
        if all(0 <= idx_resamp[i] < cine_full_fov.GetSize()[i] for i in range(3)):
            # Round to nearest integer index
            idx_resamp_int = tuple(int(round(x)) for x in idx_resamp)
            orig_vals.append(cine_frame0.GetPixel(idx))
            resampled_vals.append(cine_full_fov.GetPixel(idx_resamp_int))
    except:
        pass

orig_vals = np.array(orig_vals)
resampled_vals = np.array(resampled_vals)
correlation = np.corrcoef(orig_vals, resampled_vals)[0, 1]

print(f"Sampled {len(orig_vals)} corresponding points")
print(f"Intensity correlation: {correlation:.4f}")
print(f"Mean absolute difference: {np.mean(np.abs(orig_vals - resampled_vals)):.2f}")

Sampled 100 corresponding points
Intensity correlation: 0.9925
Mean absolute difference: 21.40


## all patients

In [21]:
import nibabel as nib


def print_pixel_spacing_and_shape(patient_id):
    patient_nifti_dir = f'/home/ayeluru/vascular-superenhancement-4d-flow/working_dir/all_patients/patient_data/{patient_id}/nifti'

    compiled_3d_cine_filepath = f'{patient_nifti_dir}/3d_cine_{patient_id}.nii.gz'
    compiled_4d_flow_filepath = f'{patient_nifti_dir}/4d_flow_mag_{patient_id}.nii.gz'

    resampled_3d_cine_dir = f'{patient_nifti_dir}/3d_cine_{patient_id}_per_timepoint'
    resampled_4d_flow_dir = f'{patient_nifti_dir}/4d_flow_mag_{patient_id}_per_timepoint'

    one_timepoint_3d_cine_filepath = f'{resampled_3d_cine_dir}/3d_cine_{patient_id}_frame_00.nii.gz'
    one_timepoint_4d_flow_filepath = f'{resampled_4d_flow_dir}/4d_flow_mag_{patient_id}_frame_00.nii.gz'

    try:
        compiled_3d_cine_nifti = nib.load(compiled_3d_cine_filepath)
        print("compiled 3d cine", compiled_3d_cine_nifti.shape, compiled_3d_cine_nifti.header.get_zooms())
    except Exception as e:
        print(f"Error loading compiled 3d cine for {patient_id}: {e}")

    try:
        compiled_4d_flow_nifti = nib.load(compiled_4d_flow_filepath)
        print("compiled 4d flow", compiled_4d_flow_nifti.shape, compiled_4d_flow_nifti.header.get_zooms())
    except Exception as e:
        print(f"Error loading compiled 4d flow for {patient_id}: {e}")

    try:
        one_timepoint_3d_cine_nifti = nib.load(one_timepoint_3d_cine_filepath)
        print("one_timepoint 3d cine", one_timepoint_3d_cine_nifti.shape, one_timepoint_3d_cine_nifti.header.get_zooms())
    except Exception as e:
        print(f"Error loading one_timepoint 3d cine for {patient_id}: {e}")
    
    try:
        one_timepoint_4d_flow_nifti = nib.load(one_timepoint_4d_flow_filepath)
        print("one_timepoint 4d flow", one_timepoint_4d_flow_nifti.shape, one_timepoint_4d_flow_nifti.header.get_zooms())
    except Exception as e:
        print(f"Error loading one_timepoint 4d flow for {patient_id}: {e}")


In [ ]:
import os

patients_dir = '/home/ayeluru/vascular-superenhancement-4d-flow/working_dir/all_patients/patient_data'

patient_ids = sorted([f.name for f in os.scandir(patients_dir) if f.is_dir()])

print(patient_ids)


['Amifer', 'Amupam', 'Apuefquor', 'Aruborn', 'Asonlig', 'Badiswu', 'Balboloop', 'Befeto', 'Bephedou', 'Bibathot', 'Bigeral', 'Biswifo', 'Bitistep', 'Bogeebo', 'Bomatog', 'Boochuto', 'Boudubat', 'Boumorim', 'Bovutou', 'Bukrukesh', 'Burapo', 'Butiswu', 'Cadedag', 'Cadotueg', 'Cefaru', 'Cemuquey', 'Ceriba', 'Ceyebum', 'Ceymuslek', 'Cisiebol', 'Coosimo', 'Cornuefor', 'Crutaswo', 'Dalibul', 'Dapafem', 'Datokif', 'Desoomi', 'Detodu', 'Diboscey', 'Diecudey', 'Diepami', 'Diequipi', 'Difresa', 'Dinaspig', 'Dithigog', 'Drusdinut', 'Dublafer', 'Dudoblo', 'Dufipat', 'Dujomal', 'Duquestank', 'Dutungub', 'Ebukhot', 'Edengat', 'Egumud', 'Ekotey', 'Elagieg', 'Emalem', 'Epcedin', 'Eposur', 'Ernegur', 'Eromax', 'Erusar', 'Eyostoy', 'Fecoostra', 'Fejoba', 'Fibrefob', 'Fisale', 'Fliesiemo', 'Fohako', 'Frahidiel', 'Frenipa', 'Fudoquo', 'Fuekeswa', 'Fumtufoos', 'Fyenpip', 'Ganage', 'Gascoyar', 'Geedefou', 'Gemapoey', 'Gepeta', 'Getahig', 'Gidusi', 'Gifranu', 'Gikolu', 'Githucu', 'Glomahe', 'Gobulou', 'Golot

In [22]:
for patient_id in patient_ids:
    print(patient_id)
    print_pixel_spacing_and_shape(patient_id)
    print("\n")


Amifer
compiled 3d cine (512, 512, 129, 20) (0.7031, 0.7031, 1.4000018, 1.0)
compiled 4d flow (256, 256, 152, 20) (1.4062508, 1.4062508, 1.399902, 1.0)
one_timepoint 3d cine (512, 512, 129) (0.7031, 0.7031, 1.4000018)
one_timepoint 4d flow (512, 512, 129) (0.7031, 0.7031, 1.4000018)


Amupam
compiled 3d cine (512, 512, 96, 20) (0.7422, 0.7422, 1.3999976, 1.0)
compiled 4d flow (256, 256, 136, 20) (1.4062508, 1.4062508, 139.98996, 1.0)
one_timepoint 3d cine (512, 512, 96) (0.7422, 0.7422, 1.3999976)
one_timepoint 4d flow (512, 512, 96) (0.7422, 0.7422, 1.3999976)


Apuefquor
compiled 3d cine (512, 512, 163, 20) (0.6641, 0.6641, 1.4000001, 1.0)
compiled 4d flow (256, 256, 170, 20) (1.3281265, 1.3281265, 1.400056, 1.0)
one_timepoint 3d cine (512, 512, 163) (0.6641, 0.6641, 1.4000001)
one_timepoint 4d flow (512, 512, 163) (0.6641, 0.6641, 1.4000001)


Aruborn
compiled 3d cine (256, 256, 119, 20) (1.5625, 1.5625, 1.5, 1.0)
compiled 4d flow (256, 256, 144, 20) (1.4062508, 1.4062508, 1.99986, 

## one patient

In [ ]:
import nibabel as nib
import os

patient_id = 'Balboloop'

patient_nifti_dir = f'/home/ayeluru/vascular-superenhancement-4d-flow/working_dir/all_patients/patient_data/{patient_id}/nifti'

compiled_3d_cine_filepath = f'{patient_nifti_dir}/3d_cine_{patient_id}.nii.gz'
compiled_4d_flow_filepath = f'{patient_nifti_dir}/4d_flow_mag_{patient_id}.nii.gz'

resampled_3d_cine_dir = f'{patient_nifti_dir}/3d_cine_{patient_id}_per_timepoint'
resampled_4d_flow_dir = f'{patient_nifti_dir}/4d_flow_mag_{patient_id}_per_timepoint'

one_timepoint_3d_cine_filepath = f'{resampled_3d_cine_dir}/3d_cine_{patient_id}_frame_00.nii.gz'
one_timepoint_4d_flow_filepath = f'{resampled_4d_flow_dir}/4d_flow_mag_{patient_id}_frame_00.nii.gz'

compiled_3d_cine_nifti = nib.load(compiled_3d_cine_filepath)
compiled_4d_flow_nifti = nib.load(compiled_4d_flow_filepath)

one_timepoint_3d_cine_nifti = nib.load(one_timepoint_3d_cine_filepath)
one_timepoint_4d_flow_nifti = nib.load(one_timepoint_4d_flow_filepath)

print("compiled 3d cine", compiled_3d_cine_nifti.shape, compiled_3d_cine_nifti.header.get_zooms())
print("compiled 4d flow", compiled_4d_flow_nifti.shape, compiled_4d_flow_nifti.header.get_zooms())

print("one_timepoint 3d cine", one_timepoint_3d_cine_nifti.shape, one_timepoint_3d_cine_nifti.header.get_zooms())
print("one_timepoint 4d flow", one_timepoint_4d_flow_nifti.shape, one_timepoint_4d_flow_nifti.header.get_zooms())






compiled 3d cine (256, 256, 70, 20) (1.4843954, 1.4844005, 1.7999979, 1.0)
compiled 4d flow (256, 256, 140, 20) (1.4062508, 1.4062508, 1.799874, 1.0)
one_timepoint 3d cine (256, 256, 70) (1.4843954, 1.4844005, 1.7999979)
one_timepoint 4d flow (256, 256, 70) (1.4843954, 1.4844005, 1.7999979)


## one file

In [ ]:
import nibabel as nib

nifti_filepath = '/home/ayeluru/vascular-superenhancement-4d-flow/working_dir/all_patients/patient_data/Balboloop/nifti/3d_cine_Balboloop.nii.gz'

In [10]:

# Load the NIfTI file
nifti_img = nib.load(nifti_filepath)

# Get the data array
data = nifti_img.get_fdata()

# Print the dimensions
print(data.shape)

# print voxel dimensions
print(nifti_img.header.get_zooms())



(256, 256, 70, 20)
(1.4843954, 1.4844005, 1.7999979, 1.0)
